# 📗 RAG 평가를 MLflow에 기록하고 인용 검증하기

지난 단원에서는 DeepEval 지표로 답변 하나하나를 채점했습니다. 이 교안은 그 채점을 골든셋(정답셋, 평가용으로 고정해 둔 질문과 정답 답변 묶음)으로 **반복하고 결과를 남기는 일**로 넓힙니다.

RAG를 고칠 때마다 사람이 질문 몇 개를 넣어 보는 방식으로는 전에 맞던 답이 틀리게 바뀐 것을 놓치기 때문입니다.

가상의 **한빛구립도서관** 이용 안내 챗봇을 골든셋으로 평가하고, 실행 조건·점수·사례별 판정 이유를 **MLflow**(머신러닝 실험 기록 도구)에 기록합니다. 이어서 답변의 인용이 원문과 맞는지 코드로 자동 확인하고, 그 검사를 채점 함수(scorer)로 만들어 평가에 넣습니다.

**학습 목표**

1. **MLflow의 역할과 기본 기록 방법을 익힙니다.** 실험, run, 파라미터, 지표, 태그, 산출물을 남기고 화면에서 찾습니다.
2. **답변 부분마다 인용을 붙이는 RAG 답변 함수를 준비합니다.** 답변을 부분(segment)으로 나누고, 문서에서 가져온 사실을 쓴 부분마다 근거 ID와 원문 구절을 붙입니다.
3. **미리 만든 골든셋을 불러와 평가 입력 형식으로 바꿉니다.** 질문마다 정답 답변을 읽고, `evaluate`가 읽는 `inputs`·`expectations` 형식으로 바꿉니다.
4. **채점 함수 세 개를 만듭니다.** DeepEval 지표를 MLflow 채점 함수로 감싼 정답 판정·검색 근거 판정과, 근거 ID 확인·인용 문구 대조를 하는 코드 채점 함수입니다.
5. **`mlflow.genai.evaluate`로 한 번에 평가하고 결과를 읽습니다.** 채점 함수별 평균, 사례별 판정과 이유, 실행 기록(trace)을 확인합니다.

<img src="images/01_eval_automation_flow.png" width="1050" alt="프롬프트 수정, 변경 요청, 자동 평가 실행, 결과 기록, 기준선과 비교를 거쳐 통과하면 병합하고 하락하면 차단하는 흐름입니다.">

이 흐름 가운데 이 교안은 **자동 평가 실행과 결과 기록**을 만듭니다. 기준선(비교 기준으로 정한 평가 run) 비교, pytest 회귀 테스트, 품질이 떨어진 변경의 병합을 막는 설정은 교안 02에서 다룹니다.

## 1. MLflow 알아보기

평가를 여러 번 돌리면 결과가 쌓이므로, 언제 어떤 설정으로 몇 점이 나왔는지 남겨 두어야 비교할 수 있습니다. 이 절에서는 그 일을 맡는 MLflow의 역할을 보고, 기록을 남기는 기본 사용법을 직접 실행해 봅니다.

### 1-1. MLflow가 맡는 일

**MLflow**는 머신러닝 실험을 기록하고 비교하는 오픈소스 도구입니다. LLM 앱에서는 평가를 반복해 실행하고 그 결과를 한곳에 남기는 데 씁니다.

<img src="images/10_mlflow_role.png" width="1050" alt="골든셋의 질문마다 답변 함수가 답하고 채점 함수가 채점하면, MLflow가 그 평가를 run 하나로 기록하고 화면에서 run끼리 비교하는 흐름입니다.">

- **평가 실행**: `mlflow.genai.evaluate`가 골든셋의 질문마다 답변 함수를 부르고, 넘겨받은 채점 함수로 채점합니다.
- **run 기록**: 평가 한 번이 run 하나로 남습니다. 실행 조건, 지표 평균, 사례별 판정과 이유가 함께 들어갑니다.
- **실행 기록(trace)**: 질문 하나가 검색과 답변 생성을 거친 단계를 남깁니다. 틀린 답이 검색 탓인지 답변 생성 탓인지 가를 때 봅니다.
- **비교**: 화면이나 코드로 run끼리 나란히 놓고 무엇이 달라졌는지 봅니다.

채점 기준은 MLflow가 정하지 않습니다. 오늘 쓰는 도구의 역할은 이렇게 나뉩니다.

| 하는 일 | 도구 |
|---|---|
| 답이 맞는지, 검색 문서에 근거하는지 채점 | DeepEval 지표(G-Eval, Faithfulness) |
| 붙인 인용이 원문과 맞는지 채점 | 직접 만드는 코드 채점 함수 |
| 평가 실행, 결과 기록, run 비교 | MLflow |
| 점수가 떨어진 변경 막기 | pytest와 CI(교안 02) |

### 1-2. 비슷한 도구와 비교

실험 기록 도구로는 MLflow 말고 **W&B**(Weights & Biases)도 많이 씁니다. 둘 다 run마다 설정과 지표를 남기고 화면에서 비교하지만, 서버를 누가 운영하느냐가 다릅니다.

| 구분 | MLflow | W&B |
|---|---|---|
| 라이선스 | 전체 오픈소스(Apache 2.0) | 파이썬 라이브러리(`wandb`)는 MIT, 서버와 화면은 비공개 상용 |
| 서버 | `mlflow server`로 누구나 직접 띄움(내 PC, 사내 서버) | 기본은 W&B 클라우드. 직접 설치(W&B Server)에는 라이선스가 필요 |
| 비용 | 무료(관리형 서비스를 쓰면 유료) | 개인·학술용 무료 등급이 있고, 팀·기업은 유료 |
| LLM 평가·추적 | `mlflow.genai.evaluate`, 실행 기록(trace) | W&B Weave |

이 과정은 계정 없이 내 PC에서 서버를 띄우고, 기록이 단원 폴더의 파일(`mlflow.db`) 하나에 남는 MLflow를 씁니다.

### 1-3. 기본 사용법: 실험, run, 파라미터, 지표

MLflow의 기록은 실험 안에 run이, run 안에 파라미터·지표·태그·산출물이 들어가는 구조입니다. 평가 run도 같은 구조라, 여기서 익힌 이름이 뒤의 평가 결과와 화면에 그대로 나옵니다.

| 이름 | 뜻 | 남기는 함수 | 예 |
|---|---|---|---|
| 기록 저장 위치 | run과 실행 기록을 저장하는 곳 | `mlflow.set_tracking_uri` | 단원 폴더의 `mlflow.db` |
| 실험(experiment) | 같은 대상의 run 묶음(챗봇 하나의 평가 기록 모음) | `mlflow.set_experiment` | `library-rag-eval`(도서관 챗봇 평가), `hr-rag-eval`(HR 챗봇 평가) |
| run | 한 번 실행한 기록. `run_name`은 표시용 이름, 구분은 `run_id` | `mlflow.start_run` | 평가 한 번(교안 02의 `top3-a` 등) |
| 파라미터(param) | 실행 조건(run마다 다를 수 있고, 한 run 안에서는 한 번 기록하면 고칠 수 없음) | `log_param`, `log_params` | `top_k=3`, 프롬프트 버전 |
| 지표(metric) | 결과 숫자 | `log_metric`, `log_metrics` | 정답 판정 평균 |
| 태그(tag) | 찾거나 구분하려고 붙이는 이름과 값 | `set_tag` | 검토 상태 |
| 산출물(artifact) | run에 붙여 두는 파일 | `log_dict`, `log_artifact` | 실패 사례 목록 |

아래 셀은 연습용 실험 `mlflow-basics`에 run 하나를 만들고 위 기록을 모두 남깁니다. 지표 값은 기록을 연습하려고 직접 적은 예시 값입니다.

In [ ]:
import mlflow
from pathlib import Path

# 1) 기록 저장 위치: 단원 폴더의 mlflow.db 파일(SQLite, 파일 하나로 된 데이터베이스)입니다. 화면 서버도 이 파일을 읽습니다.
mlflow.set_tracking_uri(f"sqlite:///{Path('.') / 'mlflow.db'}")

# 2) 실험: 같은 대상(같은 챗봇, 같은 평가)의 run 을 한데 묶습니다. 이름이 같으면 있던 실험을 다시 쓰고, 없으면 새로 만듭니다.
basics_experiment = mlflow.set_experiment("mlflow-basics")

# 3) run: 실행 한 번의 기록입니다. run_name 은 화면에 보일 표시용 이름이라 같은 이름이 여러 개일 수 있고, 구분은 run_id 로 합니다.
#    with 블록이 끝나면 run 이 닫힙니다.
with mlflow.start_run(run_name="first-run") as basics_run:
    # 실행 조건(파라미터): 하나씩 또는 dict 로 한꺼번에 남깁니다.
    mlflow.log_param("top_k", 3)
    mlflow.log_params({"prompt_version": "v1", "app_model": "gpt-4.1"})

    # 결과(지표): 숫자만 남길 수 있습니다.
    mlflow.log_metric("correctness", 0.875)
    mlflow.log_metrics({"citation_valid": 1.0, "case_count": 16})

    # 태그 하나: 이름(키)은 purpose, 값은 "기본 사용법 연습"입니다. run 을 찾거나 구분할 때 씁니다.
    mlflow.set_tag("purpose", "기본 사용법 연습")

    # 산출물: 파일로 남길 내용입니다. 실패한 사례의 case_id 와 실패 내용을 적은 예시입니다.
    failures = {"lib-12": "대출 기간을 14일로 안내", "lib-13": "근거를 찾지 못해 답을 보류"}
    # 둘째 인자는 run 의 산출물(Artifacts) 안에 새로 만들 파일 이름입니다. 확장자 .json 이라 JSON 으로 저장됩니다.
    mlflow.log_dict(failures, "failures.json")

print("실험 ID:", basics_experiment.experiment_id, "/ run_id:", basics_run.info.run_id)

닫힌 run에도 나중에 기록을 덧붙일 수 있습니다.

평가가 끝난 run에 사람이 검토한 결과를 남길 때 이 방법을 씁니다. 남긴 기록은 `get_run`과 `mlflow.artifacts.load_dict`로 다시 읽습니다.

In [ ]:
# run_id 로 같은 run 을 다시 열어 두 번째 태그(이름 review, 값 "확인함")를 덧붙입니다.
with mlflow.start_run(run_id=basics_run.info.run_id):
    mlflow.set_tag("review", "확인함")

# 저장한 파라미터·지표·태그와 산출물 파일을 다시 읽습니다.
saved_run = mlflow.get_run(basics_run.info.run_id)

print("파라미터:", saved_run.data.params)
print("지표:", saved_run.data.metrics)
print("태그 review:", saved_run.data.tags["review"])

# 산출물은 "runs:/<run_id>/파일 이름" 주소로 다시 읽습니다. 파일 이름은 log_dict 의 둘째 인자와 같습니다.
print("산출물:", mlflow.artifacts.load_dict(f"runs:/{basics_run.info.run_id}/failures.json"))
# 산출물 파일이 실제로 저장된 폴더입니다(mlruns/<실험 ID>/<run_id>/artifacts).
print("산출물 폴더:", saved_run.info.artifact_uri)

기록은 MLflow 화면에서도 봅니다.

터미널을 하나 더 열어 저장소 최상위 폴더에서 단원 폴더로 이동한 뒤 화면 서버를 켭니다. 서버는 교안을 마칠 때까지 켜 둡니다.

```bash
cd day61_평가자동화_MLflow
uv run mlflow server --backend-store-uri sqlite:///mlflow.db --port 5050
```

| 명령의 부분 | 뜻 |
|---|---|
| `uv run` | 노트북과 같은 실습 환경(가상환경)에 설치된 프로그램으로 실행합니다 |
| `mlflow server` | 기록을 읽어 브라우저 화면으로 보여 주는 MLflow 서버를 켭니다 |
| `--backend-store-uri sqlite:///mlflow.db` | 서버가 읽을 기록 저장소입니다. 지금 폴더의 `mlflow.db` 파일로, 노트북이 `set_tracking_uri`로 기록한 바로 그 파일입니다 |
| `--port 5050` | 기본값 5000은 맥의 AirPlay와 겹쳐 5050을 씁니다 |

5050을 다른 프로그램이 쓰고 있으면 다른 번호로 켜고, 아래 주소의 번호도 같이 바꿉니다.

서버는 꼭 단원 폴더에서 켭니다. 다른 폴더에서 켜면 그 폴더에 빈 `mlflow.db`가 새로 생겨 run이 보이지 않으니, `Ctrl+C`로 끄고 단원 폴더에서 다시 켭니다.

브라우저로 `http://127.0.0.1:5050`을 열고 실험 목록에서 `mlflow-basics`를 누릅니다. 처음 열리는 **Overview**는 실행 기록(trace) 요약이라 비어 있으니, 왼쪽 메뉴의 **Evaluation runs**를 누르면 `first-run`이 보입니다.

run 이름을 누르면 상세 화면이 열립니다. 지표(Metrics), 파라미터(Parameters), 태그(Tags)가 한 화면에 있고, 산출물 파일은 위쪽 **Artifacts** 탭에 있습니다.

<img src="images/ml_00_run_page.png" width="1050" alt="first-run 상세 화면. 지표 correctness, citation_valid, case_count와 파라미터 top_k, prompt_version, app_model, 태그 purpose와 review가 보입니다.">

## 2. 평가할 챗봇 준비

평가할 대상인 도서관 안내 챗봇을 만듭니다. 설정, 문서 검색, 인용을 붙이는 답변 함수 순으로 준비합니다.

### 2-1. 라이브러리·모델·MLflow 설정

지난 단원처럼 OpenAI와 DeepEval 판사(`judge_llm`)를 설정합니다. 판사 모델의 기본값은 `gpt-5.6-luna`입니다. MLflow 쪽은 두 가지입니다.

- **기록 저장 위치와 실험**: 앞의 연습과 같은 `mlflow.db`를 쓰고, 평가 run은 실험 `library-rag-eval`에 모읍니다.
- **자동 기록(autolog) 끄기**: `mlflow.langchain.autolog`는 LangChain 체인·모델 호출을, `mlflow.openai.autolog`는 OpenAI SDK 호출(판사 호출 포함)을 단계마다 입력·출력·토큰 수와 함께 실행 기록에 자동으로 남깁니다. 꺼 두지 않으면 평가(`mlflow.genai.evaluate`)가 평가 중에 이 둘을 잠시 켜는데, 이 실습의 MLflow 3.14에서는 그 기록이 늦게 닫히면 평가가 실행 기록을 찾지 못해 오류로 멈출 때가 있습니다. 그래서 끄고, 실행 기록은 `@mlflow.trace`로 직접 남깁니다.

In [ ]:
# 실습에 필요한 라이브러리를 불러옵니다.
import os
import warnings
from pathlib import Path

import mlflow
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# DeepEval 이 사용 기록을 외부로 보내지 않게 합니다. deepeval 을 불러오기 전에 정해야 합니다.
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"

# 채점 중 나오는 ipywidgets 설치 권고 경고를 숨깁니다.
warnings.filterwarnings("ignore", message='install "ipywidgets"')
from deepeval.models import OpenAIModel

# material_dir 는 data 와 mlflow.db 가 있는 단원 폴더입니다.
material_dir = Path(".")
data_dir = material_dir / "data"

load_dotenv(find_dotenv(usecwd=True))
# 단원 .env에 없는 공통 키는 저장소 최상위 .env에서 읽습니다. 이미 읽은 키는 덮어쓰지 않습니다.
load_dotenv(material_dir.resolve().parent / ".env")

# 답변 모델: 챗봇 답변을 만듭니다.
model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-4.1")
llm = ChatOpenAI(model=model_name, use_responses_api=True)

# 미리 만들어 둔 문서 벡터와 같은 모델·차원으로 질문만 임베딩합니다.
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large", dimensions=768,
                                   check_embedding_ctx_length=False)  # 짧은 질문이라 자동 길이 검사·분할을 끕니다.

# 판사 모델: 답변 모델과 다른 모델로 채점합니다. 판사를 바꾸면 기준선도 다시 만들어야 합니다.
# gpt-5.6-luna 는 temperature 를 지정할 수 없는 모델이라 지정하지 않습니다.
eval_model_name = os.getenv("OPENAI_EVAL_MODEL", "gpt-5.6-luna")
judge_llm = OpenAIModel(model=eval_model_name)

# MLflow 자동 기록(autolog)을 끕니다. 켜져 있으면 호출마다 실행 기록(trace)에 단계가 자동으로 생깁니다.
# - mlflow.langchain.autolog: LangChain 체인·프롬프트·모델 호출을 단계마다 입력, 출력, 걸린 시간, 토큰 수와 함께 기록합니다.
# - mlflow.openai.autolog: OpenAI SDK 호출(DeepEval 판사의 호출 포함)을 보낸 메시지, 모델 이름, 응답, 토큰 수와 함께 기록합니다.
# 꺼 두지 않으면 mlflow.genai.evaluate 가 평가 중에 이 둘을 잠시 켭니다. MLflow 3.14 에서는 그 기록이 늦게 닫히면 평가가 실행 기록을 찾지 못해 오류로 멈출 때가 있습니다.
# 실행 기록은 @mlflow.trace 로 직접 남깁니다.
mlflow.langchain.autolog(disable=True)
mlflow.openai.autolog(disable=True)

# 평가 run 과 실행 기록은 단원 폴더의 mlflow.db 파일에 쌓습니다.
mlflow.set_tracking_uri(f"sqlite:///{material_dir / 'mlflow.db'}")
experiment = mlflow.set_experiment("library-rag-eval")

print("답변 모델:", model_name, "/ 판사 모델:", eval_model_name, "/ 실험:", experiment.name)

### 2-2. 이용 안내 문서와 검색 함수

본 실습은 가상의 **한빛구립도서관 이용 안내** 문서 15개를 씁니다. 대출 권수·기간, 연장, 연체, 예약, 상호대차처럼 익숙한 규칙이라 답이 맞는지 바로 판단할 수 있습니다.

문서 벡터는 미리 만들어 `data/본실습/chroma/`에 넣어 두었고, 실행할 때는 질문만 임베딩합니다.

In [ ]:
import json

import pandas as pd
from langchain_chroma import Chroma

# read_jsonl: data 폴더의 jsonl 파일(한 줄에 JSON 하나)을 목록으로 읽습니다.
def read_jsonl(name):
    rows = []
    for line in (data_dir / name).read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


library_docs = read_jsonl("본실습/docs.jsonl")

# 인용을 검증할 때 근거 ID로 원문을 찾을 수 있게 사전으로 둡니다.
library_texts = {}
for doc in library_docs:
    library_texts[doc["doc_id"]] = doc["text"]

vector_store = Chroma(collection_name="library_docs", embedding_function=embedding_model,
                      persist_directory=str(data_dir / "본실습" / "chroma"))

print("이용 안내 문서:", len(library_docs))

display(pd.DataFrame(library_docs))

검색 함수에는 두 가지를 맞춰 둡니다.

- **`@mlflow.trace(span_type="RETRIEVER")`**: 이 함수의 입력과 반환 문서를 실행 기록의 **검색 단계**(span)로 남기라는 표시입니다. MLflow 화면에서 질문마다 어떤 문서를 검색했는지 볼 수 있습니다.
- **근거 ID**: 답변이 인용할 때 쓰는 문서 이름입니다. 문서 metadata의 `source_id`(docs.jsonl의 `doc_id`와 같은 값)에 넣어 두었고 `doc.metadata["source_id"]`로 읽습니다.

In [ ]:
@mlflow.trace(span_type="RETRIEVER")
def search_documents(search_query, top_k=3):
    """질문과 가까운 안내 문서 top_k 개를 Document 목록으로 반환합니다."""
    return vector_store.similarity_search(search_query, k=top_k)


def format_documents(docs):
    """검색 문서를 [근거 ID] 제목과 본문 형식으로 이어 붙입니다."""
    parts = []
    for doc in docs:
        parts.append(f"[{doc.metadata['source_id']}] {doc.metadata['title']}\n{doc.page_content}")
    return "\n\n".join(parts)

for doc in search_documents("책은 몇 권까지 빌릴 수 있나요?"):
    print(doc.metadata["source_id"], "|", doc.page_content[:50])

### 2-3. 답변 부분마다 인용을 붙이는 답변 함수

챗봇의 답변 함수 `answer_with_citations`를 만듭니다. 모델이 답변을 **답변 부분**(segment)으로 나눠 쓰고, 문서에서 가져온 사실을 쓴 부분마다 근거 ID와 그 문서에서 그대로 복사한 구절(quote)을 붙이게 합니다. 근거 ID와 구절을 합쳐 인용(citation)이라고 합니다.

LLM 서비스의 인용 기능도 대개 답변을 인용이 붙은 조각의 목록으로 돌려줍니다.

근거 ID 목록만 받으면 **어느 문장이 어느 문서의 어느 구절을 근거로 삼는지** 알 수 없습니다. 부분마다 인용을 받아 두면 인용이 맞는지 부분 단위로 검사할 수 있습니다.

도서관 예로 보면 이렇습니다.

1. 질문: "신간도 2주 동안 빌릴 수 있나요?"
2. 부분 1: "신간 코너 도서는 7일 동안 빌릴 수 있습니다." / 인용: `loan-new-books`, "1인 3권까지 7일 동안 빌릴 수 있으며"
3. 부분 2: "신간 코너 도서는 연장할 수 없습니다." / 인용: `loan-new-books`, "반납일 연장 대상에서 제외됩니다"
4. 4절에서 인용의 근거 ID와 구절이 원문과 맞는지 코드로 확인합니다.

답변 `answer`는 부분의 `text`를 순서대로 이은 것입니다. 연결 문장이나 확인할 수 없다는 안내처럼 문서의 사실이 없는 부분은 `citations`가 빈 목록입니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

class Citation(BaseModel):
    evidence_id: str = Field(description="근거 ID를 넣습니다. 제공된 ID를 철자 그대로 씁니다")
    quote: str = Field(description="그 근거 원문에서 글자 그대로 복사한 구절을 넣습니다. 고치거나 요약하지 않습니다")


class Segment(BaseModel):
    text: str = Field(description="답변의 한 부분(보통 문장 하나)을 넣습니다. segments 의 text 를 순서대로 이으면 사용자에게 보여 줄 답변이 됩니다")
    citations: list[Citation] = Field(description="이 부분을 뒷받침하는 인용 목록입니다. 문서에서 가져온 사실이 아니면(연결 문장, 확인할 수 없다는 안내 등) 빈 목록입니다")


class CitedAnswer(BaseModel):
    segments: list[Segment] = Field(description="답변을 순서대로 나눈 부분 목록입니다")


cited_prompt = ChatPromptTemplate.from_messages([
    ("system", "도서관 이용 안내 문서로만 답하세요. 답변을 부분(segments)으로 나눠 쓰고, 문서에서 가져온 사실을 쓴 부분마다 근거 ID와 "
               "그 문서에서 글자 그대로 복사한 구절을 인용하세요. 구절을 바꿔 쓰거나 요약하지 마세요. 조건과 예외가 있으면 함께 쓰세요. "
               "문서에 답이 없으면 '제공된 이용 안내에서 확인할 수 없습니다.'라는 부분 하나만 인용 없이 쓰세요."),
    ("human", "질문: {question}\n\n문서:\n{context}"),
])

cited_chain = cited_prompt | llm.with_structured_output(CitedAnswer)

답변 함수 `answer_with_citations`에는 두 가지를 맞춰 둡니다.

- **`@mlflow.trace`**: 호출 한 번이 실행 기록 하나가 되고, 안에서 부른 `search_documents`는 그 기록의 하위 단계로 남습니다.
- **dict 반환**: 채점 함수는 반환한 dict를 답변 출력(`outputs`)으로 읽습니다. 그래서 구조화 출력(`segments`)에 부분을 이은 답변(`answer`)과 이번에 검색한 문서의 근거 ID(`searched_ids`)를 더해 돌려줍니다.

In [ ]:
@mlflow.trace
def answer_with_citations(question, top_k=3):
    """질문으로 문서를 검색하고, 답변 부분마다 근거 ID 와 원문 구절을 인용으로 붙여 답합니다.

    question: 사용자 질문입니다. evaluate 의 inputs["question"] 이 이 인자로 들어옵니다.
    top_k: 검색해 올 문서 수입니다.
    반환값: dict 입니다. 채점 함수는 이 dict 를 outputs 로 받습니다.
        segments: 답변 부분 목록. 부분마다 text(문장)와 citations(근거 ID·인용 구절 목록)가 있습니다.
        answer: 부분의 text 를 이어 붙인, 사용자에게 보여 줄 답변입니다.
        searched_ids: 이번에 검색해 온 문서의 근거 ID 목록입니다.
    """
    docs = search_documents(question, top_k)
    result = cited_chain.invoke({"question": question, "context": format_documents(docs)})

    # 구조화 출력 객체를 dict 로 바꿉니다. 채점 함수는 이 dict 를 outputs 로 읽습니다.
    output = result.model_dump()

    # 부분을 순서대로 이어 사용자에게 보여 줄 답변을 만듭니다.
    texts = []
    for segment in output["segments"]:
        texts.append(segment["text"])
    output["answer"] = " ".join(texts)

    # 검색한 근거 ID도 넣어야 "검색하지 않은 문서를 인용했는지" 검사할 수 있습니다.
    output["searched_ids"] = []
    for doc in docs:
        output["searched_ids"].append(doc.metadata["source_id"])
    return output

cited_sample = answer_with_citations("신간도 2주 동안 빌릴 수 있나요?")

print("답변:", cited_sample["answer"])
for segment in cited_sample["segments"]:
    print("-", segment["text"])
    for citation in segment["citations"]:
        print("   ", citation["evidence_id"], "|", citation["quote"])
print("검색한 근거 ID:", cited_sample["searched_ids"])

## 3. 골든셋 불러오기

챗봇이 한 질문에 답하는 것을 확인했습니다. 이제 챗봇을 바꿀 때마다 같은 기준으로 채점할 골든셋을 불러옵니다.

### 3-1. 회귀와 골든셋

**회귀**(regression)는 무언가를 고친 뒤에 전에는 맞던 결과가 틀리게 되는 현상입니다. RAG에서는 프롬프트 한 문장, 검색 문서 수, 문서 버전만 바꿔도 일부 질문의 답이 달라집니다.

회귀를 찾으려면 지난 단원에서 만든 것과 같은 **골든셋**(정답셋)을 고정해 두고, 무언가를 바꿀 때마다 같은 방식으로 채점합니다.

도서관 안내 챗봇이라면 이런 일이 생깁니다.

1. 답변이 길다는 의견을 듣고 프롬프트에 "한 문장으로 답하라"를 넣습니다.
2. 대출 권수 질문은 여전히 잘 맞습니다.
3. 그런데 "상호대차로 받은 책은 연장할 수 있나요?" 질문에서 "상호대차 도서는 연장할 수 없다"는 조건이 빠집니다.
4. 질문 몇 개만 넣어 본 사람은 조건이 빠진 것을 모른 채 배포합니다. 골든셋을 매번 같은 방식으로 채점해 두면 이 누락이 점수 하락으로 드러납니다.

### 3-2. 미리 만든 골든셋 읽기

도서관 골든셋 16건은 `data/본실습/golden.jsonl`에 있습니다. 한 가지 사실을 묻는 질문, 여러 문서를 합쳐야 답하는 질문, 예외 조건을 묻는 질문, 자료에 없는 질문을 고루 담았고, 질문마다 다음 항목이 들어 있습니다.

| 필드 | 뜻 |
|---|---|
| `case_id` | 사례를 가리키는 고정 이름(예: `lib-01`) |
| `input` | 챗봇에 넣을 질문 |
| `expected_output` | 정답 답변 문장. 자료에 없는 질문이면 "제공된 이용 안내에서 확인할 수 없습니다."라는 안내 문장 |
| `context` | 정답의 근거가 되는 문서 문장 목록. 사람이 정답 답변을 원문과 대조할 때 봅니다 |
| `relevant_doc_ids` | 그 문장이 나온 문서의 근거 ID 목록(예: `["loan-basic"]`). 자료에 없는 질문이면 빈 목록 |
| `smoke` | 변경할 때마다 빠르게 점검하는 대표 사례인지(`true`/`false`). 실무에서는 골든셋이 수백 건 이상으로 커지면 변경 요청마다 이 대표 묶음만 평가해 시간과 API 비용을 줄이고, 전체는 정기 실행에서 평가합니다(교안 02) |

지난 단원에서 만든 골든셋과 같은 구조(`input`, `expected_output`, `context`)에 `case_id`, `relevant_doc_ids`, `smoke`를 더했습니다.

In [ ]:
library_golden = read_jsonl("본실습/golden.jsonl")

golden_df = pd.DataFrame(library_golden)

print("사례 수:", len(golden_df))
display(golden_df)

### 3-3. 평가 입력 형식으로 바꾸기

다음 절에서 쓸 `mlflow.genai.evaluate`는 데이터 한 행을 dict로 받고, 그 키 이름(`inputs`, `expectations` 등)은 MLflow가 정해 둔 이름이라 그대로 써야 합니다. 이름을 바꾸면(예: `input`) `evaluate`가 읽지 못해 오류가 납니다. 이 교안은 그중 두 키를 씁니다.

| 키 이름 | MLflow 규칙 | 이 교안에서 넣는 값 |
|---|---|---|
| `inputs` | 항상 있어야 합니다. 답변 함수에 키워드 인자로 넘기므로 키 이름이 답변 함수의 인자 이름과 같아야 합니다 | `{"question": 질문}` (`answer_with_citations(question)`의 인자) |
| `expectations` | 정답이 필요한 채점 함수를 쓸 때 넣습니다. 키 이름은 채점 함수가 꺼내 쓰는 이름으로 정합니다 | `{"case_id": 사례 ID, "expected_output": 정답 답변}`. 정답 판정이 `expected_output`을 읽으므로 이 교안에서는 빠지면 안 됩니다 |

`case_id`는 두 run에서 같은 질문끼리 비교할 때 행을 맞추는 데 씁니다. 평가 결과표의 행 순서가 보장되지 않기 때문입니다.

아래 `to_records`가 골든 사례를 이 형식의 목록 `library_records`로 바꿉니다.

In [ ]:
def to_records(cases):
    """골든 사례를 evaluate 가 읽는 inputs·expectations 형식으로 바꿉니다."""
    records = []
    for case in cases:
        # inputs 의 키는 답변 함수의 인자 이름(question)과 같아야 합니다.
        inputs = {"question": case["input"]}
        expectations = {"case_id": case["case_id"], "expected_output": case["expected_output"]}
        records.append({"inputs": inputs, "expectations": expectations})
    return records


library_records = to_records(library_golden)

# library_records 의 첫 행입니다. evaluate 는 행마다 이 구조를 읽습니다.
print(json.dumps(library_records[0], ensure_ascii=False, indent=1))

## 4. 채점 함수 만들기

### 4-1. 채점 함수와 `@scorer`

**채점 함수**는 답변 하나를 받아 판정(통과 여부나 점수)과 이유를 내는 함수입니다. 이 교안은 채점 함수를 세 개만 두고, 답이 맞는지·지어내지 않았는지·인용이 실제 원문인지 봅니다.

| 채점 함수 | 우리말 이름 | 판정 방법 |
|---|---|---|
| `correctness` | 정답 판정 | DeepEval G-Eval |
| `faithfulness` | 검색 근거 판정 | DeepEval Faithfulness |
| `citation_valid` | 인용 검증 | 코드 |

가장 중요한 것은 정답 판정입니다. 뜻을 읽어야 하는 판정은 LLM 판사로, 정해진 규칙으로 가릴 수 있는 판정은 비용 없이 늘 같은 결과를 내는 코드로 합니다.

이 절에서 세 채점 함수를 모두 만들고, 5절에서 `evaluate` 한 번으로 함께 평가합니다.

채점 함수의 뼈대입니다.

```python
@scorer
def 채점함수이름(outputs):
    return Feedback(value=True, rationale="판정 이유")
```

- `@scorer`: 함수를 MLflow 채점 함수로 만듭니다. 인자는 `inputs`, `outputs`, `expectations`, `trace` 중 필요한 것만 받습니다.
- `Feedback(value=..., rationale=...)`: 판정과 이유를 돌려주면 결과표에 함께 남습니다.

### 4-2. 정답 판정과 검색 근거 판정

아래 두 채점 함수는 이 뼈대 안에서 지난 단원의 DeepEval 지표(G-Eval, Faithfulness)를 불러, 통과 여부(`success`)와 이유(`reason`)를 `Feedback`으로 옮깁니다. 이름이 비슷한 RAGAS 지표와는 점수를 내는 방법이 다릅니다.

| 판정 | 이 교안(DeepEval) | RAGAS 단원 |
|---|---|---|
| 정답 판정 | G-Eval: 판사가 판단 단계(`CORRECTNESS_STEPS`)를 따라 0~1 점수를 매기고, 0.7 이상이면 통과 | Answer Correctness: 정답 답변과 사실 단위로 비교한 F1(비중 0.75)과 임베딩 유사도(비중 0.25)를 합한 점수 |
| 검색 근거 판정 | Faithfulness: 답변을 주장으로 나누고, 판사가 주장마다 `yes`(문서와 맞음), `no`(문서와 모순), `borderline`(문서에 근거 없음)을 붙여 통과한 주장의 비율이 0.7 이상이면 통과 | Faithfulness: 답변을 주장으로 나누고, 문서로 확인되는 주장의 비율 |

- **G-Eval을 쓰는 이유**: 판단 단계를 글로 적을 수 있어 "자료에 없는 질문이면 확인할 수 없다고 답해야 맞다" 같은 규칙을 넣을 수 있습니다.
- **`penalize_ambiguous_claims=True`**: `borderline` 주장도 실패로 셉니다. 기본값(`False`)은 `borderline`을 통과로 세서, 문서에 없는 말을 덧붙여도 문서와 모순만 안 되면 점수가 깎이지 않습니다. 이 챗봇은 안내 문서만 근거로 답해야 하므로 켜 둡니다. 켜면 RAGAS Faithfulness와 같은 기준이 됩니다.
- **`evaluation_template`**: DeepEval 판사 프롬프트가 영어라 Faithfulness 이유도 영어로 나옵니다. 이유를 쓰는 프롬프트(`generate_reason`)만 바꾼 `KoreanFaithfulnessTemplate`을 넘겨, 점수는 그대로 두고 이유만 한국어로 받습니다. G-Eval은 판단 단계 끝에 같은 지시를 적었습니다.

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams
from mlflow.entities import Feedback
from mlflow.genai.scorers import scorer

# 정답 판정의 판단 단계입니다. 정답 답변(expected_output)과 비교합니다.
CORRECTNESS_STEPS = [
    "expected_output 에 적힌 사실이 actual_output 에 모두 들어 있는지 본다.",
    "숫자, 기간, 대상, 조건이 하나라도 다르거나 빠지면 틀린 것으로 본다.",
    "expected_output 이 자료에서 확인할 수 없다는 안내이면, actual_output 도 확인할 수 없다고 답해야 맞다.",
    "판정 이유(reason)는 한국어로 쓴다.",
]


@scorer
def correctness(inputs, outputs, expectations):
    """정답 판정: 답변이 정답 답변(expected_output)의 사실을 모두 담았는지 G-Eval 판사로 판정해, True/False 와 이유를 Feedback 으로 돌려줍니다."""
    # 정답 답변입니다. 자료에 없는 질문이면 확인할 수 없다는 안내 문장입니다.
    expected = expectations["expected_output"]

    # 평가는 여러 질문을 동시에 채점하므로, 채점 함수가 불릴 때마다 GEval 지표를 새로 만들어 점수가 섞이지 않게 합니다.
    metric = GEval(
        name="정답 판정",
        evaluation_steps=CORRECTNESS_STEPS,
        evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT],
        model=judge_llm,
        threshold=0.7,
        async_mode=False,  # 지표 안에서 비동기로 호출하지 않습니다(동시 채점 중 이벤트 루프 오류를 막습니다).
    )

    # 여러 질문을 동시에 채점하므로 진행 표시를 끕니다(켜 두면 표시가 서로 겹칩니다).
    metric.measure(LLMTestCase(input=inputs["question"], actual_output=outputs["answer"], expected_output=expected), _show_indicator=False)
    return Feedback(value=metric.success, rationale=metric.reason)

from deepeval.metrics import FaithfulnessMetric
from deepeval.metrics.faithfulness import FaithfulnessTemplate


class KoreanFaithfulnessTemplate(FaithfulnessTemplate):
    """Faithfulness 판사 프롬프트에서 판정 이유를 쓰는 부분만 바꿔, 이유를 한국어로 받습니다."""

    @staticmethod
    def generate_reason(contradictions, score):
        # 기본 프롬프트 끝에 한국어 지시 한 줄을 붙입니다. 주장을 나누고 판정하는 프롬프트는 그대로라 점수는 같습니다.
        return FaithfulnessTemplate.generate_reason(contradictions=contradictions, score=score) + "\n\nreason 값은 한국어로 쓴다."


@scorer
def faithfulness(inputs, outputs):
    """검색 근거 판정: 답변이 이번에 검색해 온 문서로 뒷받침되는지 Faithfulness 판사로 판정해, True/False 와 이유를 Feedback 으로 돌려줍니다."""
    # 이번에 검색해 온 문서의 원문 전체를 근거(retrieval_context)로 줍니다.
    retrieval_context = []
    for doc_id in outputs["searched_ids"]:
        retrieval_context.append(library_texts[doc_id])

    # penalize_ambiguous_claims=True: 문서에 근거가 없는 주장(borderline)도 실패로 셉니다.
    # 기본값 False 는 문서와 모순되는 주장만 실패로 세서, 문서에 없는 말을 덧붙여도 통과합니다.
    # evaluation_template: 판정 이유를 한국어로 받는 프롬프트를 씁니다.
    metric = FaithfulnessMetric(model=judge_llm, threshold=0.7, async_mode=False, penalize_ambiguous_claims=True,
                                evaluation_template=KoreanFaithfulnessTemplate)

    # 여러 질문을 동시에 채점하므로 진행 표시를 끕니다(켜 두면 표시가 서로 겹칩니다).
    metric.measure(LLMTestCase(input=inputs["question"], actual_output=outputs["answer"], retrieval_context=retrieval_context), _show_indicator=False)
    return Feedback(value=metric.success, rationale=metric.reason)

### 4-3. 인용을 따로 검사하는 이유

앞의 두 채점 함수는 답의 내용을 봅니다. 세 번째 채점 함수 `citation_valid`는 답변에 붙은 인용이 맞는지 봅니다.

검색 근거 판정(`faithfulness`)은 답이 **검색해 온 문서 전체**로 뒷받침되는지만 보고, 답변에 붙은 인용은 보지 않습니다. 답의 내용이 검색한 문서로 뒷받침되면 인용을 검색하지도 않은 문서 Z로 붙여도 통과하므로, 사용자에게 보여 주는 인용이 맞는지는 따로 검사해야 알 수 있습니다.

<img src="images/07_grounding_vs_citation.png" width="1050" alt="답의 내용은 검색한 문서로 뒷받침되지만 답변이 검색하지 않은 문서 Z를 인용한 경우, 검색 문서 전체를 보는 검색 근거 판정은 통과하고 붙인 인용만 보는 인용 검증은 실패합니다.">

인용은 비용이 적게 드는 확인부터 차례로 검사합니다. 앞 단계에서 실패한 인용은 뒤 단계로 보내지 않습니다.

<img src="images/03_citation_check_layers.png" width="1050" alt="근거 ID 확인과 인용 문구 대조를 차례로 통과한 인용만 검증된 인용으로 남는 과정입니다. 두 단계 모두 코드로 검사합니다.">

1. **근거 ID 확인**: 인용한 근거 ID가 문서 목록에 있고, 이번 검색에서 가져온 문서의 ID인가
2. **인용 문구 대조**: 인용 구절이 그 문서 원문에 글자 그대로 들어 있는가

두 단계 모두 코드로 검사해 비용이 들지 않으므로 채점 함수 `citation_valid`로 만들어 평가에 넣습니다.

인용한 원문이 주장과 뜻까지 맞는지는 보지 않습니다. 답이 틀렸으면 정답 판정에서 드러나므로, 그때 실행 기록에서 인용을 열어 원인을 찾습니다.

### 4-4. 근거 ID와 인용 문구 대조

2단계는 모델이 문장을 한 글자라도 바꿔 옮긴 것을 잡습니다. 연속 공백·줄바꿈·가운뎃점 모양(`·` `･` `∙`)처럼 뜻과 상관없는 차이는 불일치로 세지 않도록, 양쪽을 같은 규칙으로 **정규화**(표기 차이를 한 가지로 맞추기)한 뒤 인용 구절이 원문 안에 들어 있는지 봅니다.

유사도로 판정하지 않는 이유는 "14일"과 "30일"처럼 숫자 하나만 다른 구절도 비슷하다고 통과하기 때문입니다.

In [ ]:
def normalize(text):
    """뜻과 상관없는 표기 차이(가운뎃점 모양, 줄바꿈, 연속 공백)를 하나로 맞춥니다."""
    for mark in ["･", "∙", "・", "•"]:
        text = text.replace(mark, "·")

    # split 은 줄바꿈·탭·연속 공백에서 모두 나누므로, 공백 하나로 다시 이으면 공백이 통일됩니다.
    return " ".join(text.split())


def check_citation(citation, source_texts, searched_ids):
    """인용 하나가 이번 답변의 실제 근거인지 차례로 검사해, 처음 걸린 문제 또는 "확인"을 돌려줍니다.

    citation: 답변 부분에 붙은 인용 하나입니다. {"evidence_id": 근거 ID, "quote": 인용 구절}
    source_texts: 근거 ID -> 문서 원문 사전입니다.
    searched_ids: 이번 질문에서 검색해 온 문서의 근거 ID 목록입니다.
    반환값: 아래 순서로 검사해 처음 걸린 문제를 돌려줍니다. 모두 통과하면 "확인"입니다.
        "없는 근거 ID": 문서에 없는 근거 ID 를 인용함
        "검색하지 않은 문서": 있는 문서지만 이번 질문에서 검색하지 않음
        "원문에 없는 문구": 표기를 맞춘(normalize) 뒤에도 인용 구절이 원문에 없음
    """
    cited_id = citation["evidence_id"]
    if cited_id not in source_texts:
        return "없는 근거 ID"
    if cited_id not in searched_ids:
        return "검색하지 않은 문서"
    if normalize(citation["quote"]) not in normalize(source_texts[cited_id]):
        return "원문에 없는 문구"
    return "확인"

검사 코드가 실제로 잘못을 잡는지는 **일부러 틀린 인용**을 넣어 봐야 압니다.

사람이 쓴 답변 하나에 세 가지 잘못을 일부러 넣었습니다. 이 답변은 모델 출력이 아니라 검사 코드를 시험하려고 만든 입력입니다.

In [ ]:
planted = {
    "searched_ids": ["loan-basic", "loan-extension"],
    "segments": [
        # 정상: quote 는 loan-basic 원문 구절을 그대로 옮김. 원문의 공백 자리에 줄바꿈이 들어간 차이만 있고, 정규화로 맞춰집니다.
        {"text": "일반 도서는 1인 7권까지 14일 동안 빌릴 수 있습니다.",
         "citations": [{"evidence_id": "loan-basic", "quote": "일반 도서를 1인 7권까지\n14일 동안 빌릴 수 있습니다"}]},
        # 잘못 1(evidence_id): loan-weekend 는 문서 목록(docs.jsonl)에 없는 근거 ID 입니다.
        {"text": "주말에는 3일 더 빌릴 수 있습니다.",
         "citations": [{"evidence_id": "loan-weekend", "quote": "주말에는 3일 더 빌릴 수 있습니다"}]},
        # 잘못 2(evidence_id): reading-room 은 있는 문서지만 이번 검색 결과(searched_ids)에 없습니다.
        {"text": "열람실은 4시간 단위로 예약합니다.",
         "citations": [{"evidence_id": "reading-room", "quote": "4시간 단위"}]},
        # 잘못 3(quote): loan-basic 원문의 "14일"을 quote 에서 "30일"로 바꿔 옮겼습니다(text 도 30일).
        {"text": "일반 도서는 30일 동안 빌릴 수 있습니다.",
         "citations": [{"evidence_id": "loan-basic", "quote": "일반 도서를 1인 7권까지 30일 동안 빌릴 수 있습니다"}]},
    ],
}

for segment in planted["segments"]:
    for citation in segment["citations"]:
        status = check_citation(citation, library_texts, planted["searched_ids"])
        print(status, "|", segment["text"])

각 줄의 `|` 앞 결과는 모델 호출 없이 코드가 정하므로 늘 `확인`, `없는 근거 ID`, `검색하지 않은 문서`, `원문에 없는 문구` 순서입니다.

### 4-5. 인용 검증 채점 함수

두 단계 검사를 `@scorer` 코드 채점 함수 `citation_valid`로 감쌉니다. 답변 부분에 붙은 인용이 모두 두 단계를 통과하면 참입니다.

`citation_valid`는 붙은 인용이 맞는지만 보고, 검색 문서로 뒷받침되지 않는 내용이 답변에 있는지는 `faithfulness`가 봅니다. 인용을 하나도 붙이지 않은 답변도 `citation_valid`를 통과하므로, 교안 02에서 기준선을 정할 때는 `correctness`도 함께 보고 사람이 표본을 확인합니다.

In [ ]:
@scorer
def citation_valid(outputs):
    """인용 검증: 답변에 붙은 인용이 모두 check_citation 을 통과하면 True, 아니면 False 와 걸린 인용 목록을 Feedback 으로 돌려줍니다."""
    problems = []
    for segment in outputs["segments"]:
        for citation in segment["citations"]:
            status = check_citation(citation, library_texts, outputs["searched_ids"])
            if status != "확인":
                problems.append(f"{status}: {citation['evidence_id']}")

    if len(problems) == 0:
        return Feedback(value=True, rationale="붙은 인용이 모두 근거 ID·문구 대조를 통과함")
    return Feedback(value=False, rationale="; ".join(problems))

## 5. `mlflow.genai.evaluate`로 평가하고 기록하기

`evaluate(data=..., predict_fn=..., scorers=...)`는 데이터의 행마다 `predict_fn(**inputs)`를 불러 답을 만들고, 채점 함수(scorer) 목록으로 그 답을 채점해 결과를 MLflow **run** 하나에 기록합니다.

### 5-1. 평가 실행

16건을 평가합니다. 질문마다 답변을 한 번 만들고 판사 모델을 여러 번 부릅니다(Faithfulness는 주장 추출과 대조에 여러 번 부릅니다).

끝나면 MLflow가 run 이름(Run name)과 ID(Run ID)를 출력하고, 그 아래에 채점 함수별 지표가 출력됩니다.

In [ ]:
# 행마다 answer_with_citations 로 답을 만들고 세 채점 함수로 채점한 뒤, 결과를 run 하나에 기록합니다.
result = mlflow.genai.evaluate(
    data=library_records,
    predict_fn=answer_with_citations,
    scorers=[correctness, faithfulness, citation_valid],
)

for name, value in result.metrics.items():
    print(name, round(float(value), 3))

### 5-2. 결과 읽기

지표 이름은 `채점함수이름/mean`입니다. 채점 함수 이름은 `@scorer`를 붙인 함수 이름(`correctness`, `faithfulness`, `citation_valid`)이고, 참(`True`)은 1, 거짓(`False`)은 0으로 세어 평균을 냅니다.

사례별 판정은 평가 결과표 `result.result_df`에 있습니다. 원래 모양부터 봅니다.

In [ ]:
# 평가 결과표의 앞 3행입니다. 한 행이 골든 사례 하나이고, 전체는 16행입니다.
display(result.result_df.head(3))

결과표의 주요 열입니다.

| 열 | 내용 |
|---|---|
| `이름/value`, `이름/rationale` | 채점 함수마다 생기는 판정과 이유(예: `correctness/value`) |
| `case_id/value`, `expected_output/value` | `expectations`에 넣은 값 |
| `request` | 평가에 넣은 `inputs`(질문) |
| `response` | 답변 함수가 돌려준 dict(답변, 인용, 검색한 근거 ID) |
| `trace`, `spans`, `assessments` | 실행 기록과 그 안의 단계, 판정 기록 |

열이 많아 사례별 판정을 보려면 필요한 열만 골라야 합니다. 행 순서도 입력 순서와 다를 수 있어 `case_id`로 정렬해 읽습니다.

In [ ]:
# case_id 와 채점 함수별 판정 열만 고르고, 행 순서가 보장되지 않으므로 case_id 순으로 정렬합니다.
judgments = result.result_df[["case_id/value", "correctness/value", "faithfulness/value", "citation_valid/value"]]
display(judgments.sort_values("case_id/value").reset_index(drop=True))

In [ ]:
# 정답 판정을 통과하지 못한 사례는 질문, 정답 답변, 챗봇 답변, 판정 이유를 함께 읽어야 원인을 알 수 있습니다.
# request 열에는 평가에 넣은 inputs(질문), response 열에는 답변 함수가 돌려준 dict 가 들어 있습니다.
for _, row in result.result_df.sort_values("case_id/value").iterrows():
    if row["correctness/value"] == False:
        print(row["case_id/value"], "| 질문:", row["request"]["question"])
        print("   정답 답변:", row["expected_output/value"])
        print("   챗봇 답변:", row["response"]["answer"])
        print("   판정 이유:", row["correctness/rationale"])

판정 이유를 정답 답변·챗봇 답변과 나란히 읽습니다. 챗봇 답변에 정답 답변의 사실이 들어 있는데도 `False`라면 판사가 잘못 판정한 것입니다.

그래서 판사의 판정은 사람이 주기적으로 일부를 뽑아 검토합니다.

In [ ]:
# 검색 근거 판정을 통과하지 못한 사례의 답변과 판정 이유를 출력합니다.
for _, row in result.result_df.sort_values("case_id/value").iterrows():
    if row["faithfulness/value"] == False:
        print(row["case_id/value"], "| 질문:", row["request"]["question"])
        print("   챗봇 답변:", row["response"]["answer"])
        print("   판정 이유:", row["faithfulness/rationale"])

아무것도 출력되지 않으면 모든 답변이 검색해 온 문서로 뒷받침된 것입니다.

자료에 없는 질문(`lib-15`, `lib-16`)에 "제공된 이용 안내에서 확인할 수 없습니다."라고 바르게 답했는데도 `False`가 나올 때가 있습니다. 판사가 이 문장을 "검색한 문서의 내용까지 확인할 수 없다"는 주장으로 잘못 읽은 것이라, 정답 판정이 `True`이고 이유가 이런 내용이면 판사 오판으로 처리합니다.

In [ ]:
# 인용 검증을 통과하지 못한 사례만 이유를 출력합니다.
for _, row in result.result_df.sort_values("case_id/value").iterrows():
    if row["citation_valid/value"] == False:
        print(row["case_id/value"], "|", row["citation_valid/rationale"])

아무것도 출력되지 않으면 이번 답변에 붙은 인용이 모두 원문과 맞은 것입니다. 출력된 줄이 있으면 이유로 실패 종류를 가립니다(`없는 근거 ID`는 없는 ID를 적은 것, `검색하지 않은 문서`는 검색 결과 밖의 문서를 인용한 것, `원문에 없는 문구`는 구절을 바꿔 옮긴 것).

자료에 없는 질문에는 인용 없는 답변 부분 하나로 답하도록 프롬프트에 지시했으므로 `citation_valid`는 참이고, 제대로 모른다고 답했는지는 `correctness`가 판정합니다. 답할 수 있는 질문에 답을 보류해도 `citation_valid`는 참이지만, 정답 답변의 사실이 빠져 `correctness`가 `False`로 드러납니다.

`faithfulness`는 통과했는데 `citation_valid`가 실패한 사례는, 답의 내용은 검색 문서로 뒷받침되지만 사용자에게 보여 준 인용(근거 ID나 구절)이 틀린 것입니다.

### 5-3. MLflow 화면에서 확인

평가 run과 실행 기록도 같은 `mlflow.db`에 저장되었으므로, 앞에서 켠 화면 서버에서 바로 봅니다(서버를 껐다면 같은 명령으로 다시 켭니다). 브라우저로 `http://127.0.0.1:5050`을 열고 실험 목록에서 `library-rag-eval`을 누릅니다.

왼쪽 메뉴의 **Evaluation runs**를 누르면 평가 run이 한 줄에 하나씩 나오고, 채점 함수별 평균(`correctness/mean` 등)이 열로 보입니다.

아래 첫 화면은 교안 02까지 실행한 뒤 찍어서 지금보다 run과 지표 열이 더 많습니다. 두 번째 화면은 실행 기록 하나의 상세 화면이라 Assessments에 세 채점 함수의 판정이 있습니다.

<img src="images/ml_01_evaluation_runs.png" width="1050" alt="평가 run마다 채점 함수별 지표를 열로 보여 주는 MLflow Evaluation runs 화면입니다.">

- 누를 곳: 실험 `library-rag-eval` > 왼쪽 메뉴 Evaluation runs
- 주소: `http://127.0.0.1:5050/#/experiments/<실험 ID>/evaluation-runs`

run 이름을 누르면 오른쪽에 질문마다 한 줄씩 실행 기록이 나오고, 실행 기록 ID를 누르면 상세 화면이 열립니다. 상세 화면의 가운데에는 입력(`question`), 검색 단계(`search_documents was called`), 출력이 있고, 오른쪽 **Assessments**에 채점 함수별 판정이 붙어 있습니다.

판정 옆 화살표를 누르면 판정 이유(Rationale)가 보이며, `result_df`의 `correctness/rationale` 열과 같은 내용입니다.

<img src="images/ml_02_trace_assessment.png" width="1050" alt="실행 기록 하나의 입력, 검색 단계, 출력과 채점 함수별 판정·판정 이유를 보여 주는 상세 화면입니다.">

- 누를 곳: Evaluation runs > run 이름 > 실행 기록 ID(`tr-...`) > Assessments 의 `correctness` 판정 옆 화살표
- 주소: `http://127.0.0.1:5050/#/experiments/<실험 ID>/evaluation-runs?selectedRunUuid=<run_id>`

`<실험 ID>`는 화면에서 실험을 누르면 주소에 나오고, `<run_id>`는 5-1절 평가 셀이 출력한 Run ID입니다.

## 6. 🖐️ 함께 따라하기: HR 유연근무 상담 챗봇 평가

본 실습에서 만든 답변 형식, 골든셋 변환, 채점 함수를 실제 공개 문서로 만든 상담 챗봇에 그대로 적용합니다.

인사팀이 사내 상담 챗봇의 검색 자료로 고용노동부 「2024 유연근무 활용 매뉴얼」을 넣었습니다. 직원이 "선택근무제는 누구에게 적용할 수 없나요?"처럼 물으면 매뉴얼을 근거로 답합니다.

배포 전에 다음을 확인하려 합니다.

- 답변이 정답 답변의 사실을 담는가
- 붙인 인용이 매뉴얼 원문과 맞는가
- 매뉴얼에 없는 질문에는 지어내지 않고 모른다고 답하는가

매뉴얼을 절 단위로 나눈 문서 44개와 골든셋 24건(매뉴얼에 없는 질문 4건 포함)이 `data/따라하기/`에 있습니다.

**실습 진행 방법**

- **직접 설계:** 클래스·함수 이름과 입출력 규약을 보고 핵심 로직 전체를 작성합니다.
- **부분 작성:** 제공한 보조 코드의 `...`를 채웁니다. 환경 준비·데이터·긴 답변 프롬프트·결과 출력은 활용하세요.
- 작성 셀을 완성한 뒤 아래 실행·출력 셀로 확인합니다. `...`는 실행 가능한 기본값이 아닙니다.
- 완성 코드는 수업이 끝난 뒤 강사가 공유합니다. 먼저 자신의 구현으로 각 절을 완성해 봅니다.

본 실습에서 만든 `read_jsonl`, `to_records`, `format_documents`, `CitedAnswer`, `check_citation`, `correctness`를 그대로 씁니다.

### 6-1. 매뉴얼과 검색 함수 준비

매뉴얼 문서·골든셋·검색 함수를 준비합니다. 골든 사례는 본 실습의 `to_records`를 써서 평가 입력 형식인 `hr_records`로 바꿉니다.

이후의 평가 run은 `hr-rag-eval` 실험에 기록합니다.

In [ ]:
hr_docs = read_jsonl("따라하기/docs.jsonl")
hr_golden = read_jsonl("따라하기/golden.jsonl")
hr_records = to_records(hr_golden)

hr_texts = {}
for doc in hr_docs:
    hr_texts[doc["doc_id"]] = doc["text"]

hr_store = Chroma(collection_name="hr_manual", embedding_function=embedding_model,
                  persist_directory=str(data_dir / "따라하기" / "chroma"))

hr_experiment = mlflow.set_experiment("hr-rag-eval")

print("매뉴얼 문서:", len(hr_docs), "/ 골든 사례:", len(hr_golden), "/ 실험:", hr_experiment.name)
display(pd.DataFrame(hr_golden).head(8))

매뉴얼 검색 함수 `search_manual`은 본 실습의 `search_documents`와 같은 모양이고, 매뉴얼 문서를 4개 가져옵니다.

In [ ]:
@mlflow.trace(span_type="RETRIEVER")
def search_manual(search_query, top_k=4):
    """매뉴얼에서 질문과 가까운 문서 top_k 개를 반환합니다."""
    return hr_store.similarity_search(search_query, k=top_k)

### 6-2. 인용을 붙이는 상담 답변 함수 (`hr_answer`)

**부분 작성 · 검색과 답변 dict 만들기**

**제공:** 매뉴얼용 프롬프트 `hr_prompt`와 체인 `hr_chain`입니다. 출력 형식은 본 실습의 `CitedAnswer`를 그대로 씁니다.

**작성:** `hr_answer(question)`의 본문을 완성하세요.

1. `search_manual(question)`으로 매뉴얼 문서 4개를 가져옵니다.
2. `format_documents`로 프롬프트의 `context`에 넣을 글을 만들어 `hr_chain`을 한 번 호출합니다.
3. 결과를 `model_dump()`로 dict로 바꾸고, `segments`의 `text`를 순서대로 공백으로 이은 `answer`와 `searched_ids`(검색한 문서의 근거 ID 목록)를 넣어 반환합니다. 형식은 본 실습의 `answer_with_citations`와 같습니다.

In [ ]:
hr_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 회사 인사팀 상담 챗봇입니다. 고용노동부 유연근무 활용 매뉴얼 발췌로만 답하세요. "
               "답변을 부분(segments)으로 나눠 쓰고, 매뉴얼에서 가져온 사실을 쓴 부분마다 근거 ID와 그 절에서 글자 그대로 복사한 구절을 인용하세요. "
               "구절을 바꿔 쓰거나 요약하지 마세요. 적용 대상, 요건, 기간, 예외를 빠뜨리지 마세요. "
               "매뉴얼에 답이 없으면 '제공된 매뉴얼에서 확인할 수 없습니다.'라는 부분 하나만 인용 없이 쓰세요."),
    ("human", "질문: {question}\n\n매뉴얼 발췌:\n{context}"),
])

hr_chain = hr_prompt | llm.with_structured_output(CitedAnswer)


@mlflow.trace
def hr_answer(question):
    """매뉴얼을 검색해 hr_chain 으로 답하고, answer_with_citations 와 같은 dict(segments, answer, searched_ids)를 돌려줍니다."""
    # 검색 -> 체인 호출 -> dict 로 바꾸고 answer(부분 text 잇기)와 searched_ids 넣기를 작성하세요.
    ...


hr_sample = hr_answer(hr_golden[0]["input"])

print("질문:", hr_golden[0]["input"])
print("답변:", hr_sample["answer"])
for segment in hr_sample["segments"]:
    print("-", segment["text"])
    for citation in segment["citations"]:
        print("   ", citation["evidence_id"], "|", citation["quote"])

### 6-3. 매뉴얼 인용 검증 채점 함수 (`hr_citation_valid`)

**직접 설계 · `hr_citation_valid`**

- `hr_citation_valid(outputs)`: 모든 답변 부분(`outputs["segments"]`)에 붙은 인용이 `check_citation(인용, hr_texts, outputs["searched_ids"])`에서 `"확인"`이면 `Feedback(value=True, ...)`를 돌려줍니다. 아니면 문제 목록을 `rationale`에 담아 `False`를 돌려줍니다.
- `@scorer`를 붙입니다. 규칙은 본 실습의 `citation_valid`와 같고, 원문 사전만 `hr_texts`로 바꿉니다.

매뉴얼 원문은 PDF에서 옮긴 글이라 `이 직 률`처럼 글자 사이에 공백이 남은 곳이 있어, 모델이 `이직률`로 붙여 쓰면 문구 대조에 실패할 수 있습니다. 이런 실패가 나오면 인용 구절과 원문(`hr_texts[근거 ID]`)을 나란히 출력해 원문 표기 문제인지 인용 오류인지 구분합니다.

In [ ]:
@scorer
def hr_citation_valid(outputs):
    """인용 검증(HR): citation_valid 와 같은 규칙으로, 원문 사전만 hr_texts 를 써서 판정합니다."""
    ...


# 앞에서 만든 hr_sample 로 먼저 검사합니다. LLM 판사를 부르지 않아 비용이 들지 않습니다.
check = hr_citation_valid(outputs=hr_sample)

print(check.value, "|", check.rationale)

### 6-4. 평가 run 기록과 확인

**부분 작성 · 평가 실행**

**작성:** `mlflow.genai.evaluate`로 24건을 평가해 `hr_result`에 담으세요.

1. `data`에는 평가 입력 형식으로 바꾼 HR 골든셋을 넘깁니다.
2. `predict_fn`에는 앞에서 만든 상담 답변 함수를 넘깁니다.
3. `scorers`에는 본 실습의 `correctness`와 앞에서 만든 `hr_citation_valid`를 넘깁니다.

검색 근거 판정은 판사 호출이 많아 따라하기에서는 넣지 않습니다.

In [ ]:
hr_result = mlflow.genai.evaluate(
    data=...,
    predict_fn=...,
    scorers=[...],
)

In [ ]:
for name, value in hr_result.metrics.items():
    print(name, round(float(value), 3))

# 사례별 판정을 case_id 순으로 봅니다.
hr_judgments = hr_result.result_df[["case_id/value", "correctness/value", "hr_citation_valid/value"]]
display(hr_judgments.sort_values("case_id/value").reset_index(drop=True))

`hr-21`~`hr-24`(매뉴얼에 없는 질문) 행은 `correctness`가 `True`이면 지어내지 않고 모른다고 답한 것입니다. 답변 출력은 `hr_result.result_df`의 `response` 열에서 보며, 이런 답변은 보통 인용 없는 부분 하나입니다.

`hr_citation_valid`가 `False`인 행은 `hr_citation_valid/rationale` 열에서 이유를 읽고, MLflow 화면의 `hr-rag-eval` 실험에서 실행 기록과 함께 확인합니다.

## 7. (선택) Jev로 같은 판정 다시 하기

지난 단원 부록의 **Jev**는 답변을 생성하지 않고 판정 문장(`Noul`)이 참일 확률을 돌려주는 판사 모델이라, 반복 평가나 CI처럼 채점을 자주 돌릴 때 쓸 만합니다. 이 절은 Jev 키(`TYPESAFE_API_KEY`)가 있을 때만 실행합니다(`.env.example` 참고).

답변은 다시 만들지 않습니다. 처음 평가한 `result`의 실행 기록을 `mlflow.search_traces(run_id=...)`로 불러와 `evaluate(data=...)`에 넘기면, 저장된 질문·답변·정답 정보로 채점만 다시 합니다(`predict_fn`을 넘기지 않습니다).

In [ ]:
from deepeval.metrics import JevEval
from deepeval.metrics.jev_eval import Noul

# Jev 모델 버전을 고정합니다(환경변수 TYPESAFE_MODEL).
jev_model_name = os.getenv("TYPESAFE_MODEL", "jev-1.13.0")


@scorer
def correctness_jev(inputs, outputs, expectations):
    """정답 판정(Jev): correctness 와 같은 판정을 Jev 판사로 해, True/False 와 이유를 Feedback 으로 돌려줍니다."""
    # 정답 답변은 correctness 와 같게 expected_output 을 씁니다.
    expected = expectations["expected_output"]

    # 판정 문장이 참일 확률이 0.5 이상이면 통과입니다(strict_mode).
    judge = JevEval(
        name="정답 판정(Jev)",
        evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT],
        questions=[Noul("actual_output은 expected_output에 적힌 사실을 숫자와 조건까지 모두 담고 있다.")],
        system_one_model=jev_model_name,
        strict_mode=True,
        async_mode=False,  # 정답 판정 채점 함수와 같은 설정입니다.
    )

    # 여러 질문을 동시에 채점하므로 진행 표시를 끕니다(켜 두면 표시가 서로 겹칩니다).
    judge.measure(LLMTestCase(input=inputs["question"], actual_output=outputs["answer"], expected_output=expected), _show_indicator=False)
    return Feedback(value=judge.success, rationale=judge.reason)

In [ ]:
# 따라하기에서 HR 실험으로 바꿨으므로 도서관 실험으로 돌아옵니다.
mlflow.set_experiment("library-rag-eval")

# 처음 평가(result)가 남긴 실행 기록 16건을 불러와 채점만 다시 합니다. 답변을 새로 만들지 않습니다.
# search_traces 는 방금 설정한 library-rag-eval 실험에서 실행 기록을 찾습니다.
traces = mlflow.search_traces(run_id=result.run_id)
jev_result = mlflow.genai.evaluate(data=traces, scorers=[correctness_jev])

print("Jev 지표:", jev_result.metrics)

In [ ]:
# 두 판사의 판정을 실행 기록 ID 로 맞춰 나란히 놓습니다.
geval_table = result.result_df[["trace_id", "case_id/value", "correctness/value"]]
jev_table = jev_result.result_df[["trace_id", "correctness_jev/value"]]
both = geval_table.merge(jev_table, on="trace_id").sort_values("case_id/value")

display(both.drop(columns="trace_id").reset_index(drop=True))

same = both["correctness/value"] == both["correctness_jev/value"]
print("판정이 같은 사례:", int(same.sum()), "/", len(both))

판정이 다른 사례가 있으면 `result.result_df`의 `response` 열에서 그 사례의 답변을 찾아 읽고 어느 판사가 맞았는지 사람이 정합니다.

Jev로 바꿔 쓰려면 기존 판사와 판정이 다른 사례를 사람이 검토해 Jev가 맞는지 먼저 확인하고, 교안 02의 기준선도 같은 판사로 다시 만듭니다. 판사가 바뀌면 이전 점수와 비교할 수 없기 때문입니다.

## 8. 핵심 정리

- **MLflow:** 실험 안에 run을 열고 파라미터(`log_param`), 지표(`log_metric`), 태그(`set_tag`), 산출물(`log_dict`)을 남깁니다. 채점은 DeepEval 지표와 코드 채점 함수가 하고, MLflow는 평가 실행·결과 기록·run 비교를 맡습니다.
- **인용을 붙이는 답변 함수:** 답변을 부분(segment)으로 나눠 문서의 사실을 쓴 부분마다 근거 ID와 원문 구절을 붙이고, 부분을 이은 답변(`answer`), 검색한 근거 ID(`searched_ids`)와 함께 dict로 돌려줍니다.
- **골든셋:** 미리 만든 골든셋을 고정해 두고 챗봇을 바꿀 때마다 같은 방식으로 채점합니다. `to_records`로 `inputs`와 `expectations`로 나눠 `evaluate`에 넘깁니다.
- **`mlflow.genai.evaluate`와 채점 함수:** 행마다 답을 만들고 세 채점 함수(정답 판정, 검색 근거 판정, 인용 검증)로 채점해 run 하나에 지표, 사례별 판정과 이유, 실행 기록을 남깁니다. 뜻을 읽어야 하는 판정은 DeepEval 지표를 `@scorer`로 감싸고, 정해진 규칙은 코드 채점 함수로 합니다.
- **인용 검사 2단계:** 붙은 인용의 근거 ID 확인과 정규화 후 문구 대조를 코드 채점 함수 `citation_valid`로 만들어 평가에 넣습니다. 인용한 원문이 주장과 뜻까지 맞는지는 보지 않고, 답이 틀리면 정답 판정에서 드러납니다.
- **다시 채점하기(선택 절):** `mlflow.search_traces(run_id=...)`로 불러온 실행 기록을 `evaluate(data=...)`에 넘기면 답변을 다시 만들지 않고 채점만 다시 합니다.

화면 서버는 다 쓴 뒤 터미널에서 `Ctrl+C`로 끕니다.

다음 교안에서는 팀이 승인한 평가 run을 **기준선**으로 정하고, 새 설정의 결과가 기준선보다 나빠지면 pytest와 CI가 변경을 막도록 연결합니다.